# **Basics of RNN Workflow**
**Steps** :
1. Vectorization
2. Embedding
3. RNN
4. Output

# Step 1: Vectorization


In [5]:
texts = [
    "keval solanki",
    "i love rnns",
    "i am learning deep learning",
    "deep learning is fun",
]

In [6]:
from keras import layers, Sequential

VOCAB_SIZE = 1000       # Max vocabulary count (distinct words allowed)
MAX_LEN = 8            # Uniform sequence length (forces text to exactly 8 words) / auto padding
EMBEDDING_DIM = 16      # Dimensions per word vector (Feature Dimension)
RNN_UNITS = 32          # Size of the internal hidden state / memory vector


vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    standardize='lower_and_strip_punctuation',
    output_sequence_length=MAX_LEN
)

vectorizer.adapt(texts)

In [7]:
vectorizer.get_vocabulary()

['',
 '[UNK]',
 np.str_('learning'),
 np.str_('i'),
 np.str_('deep'),
 np.str_('solanki'),
 np.str_('rnns'),
 np.str_('love'),
 np.str_('keval'),
 np.str_('is'),
 np.str_('fun'),
 np.str_('am')]

In [10]:
output = vectorizer("i am keval solanki")
print(output)

tf.Tensor([ 3 11  8  5  0  0  0  0], shape=(8,), dtype=int64)


# Step 2: Embeddings

In [31]:
import tensorflow as tf

# Expand dimensions to add the batch size: shape goes from (8,) to (1, 8)
batched_output = tf.expand_dims(output, axis=0)


embedding = layers.Embedding(
  input_dim = VOCAB_SIZE,       # Size of the vocabulary.
  output_dim= EMBEDDING_DIM,                 # Dimension of the dense embedding vector
  # embeddings_regularizer=keras.regularizers.L2(1e-4),    # prevent the weights from growing too large, helping reduce overfitting
)

embedding.build(input_shape=(1,8))

print(embedding.call(output))

tf.Tensor(
[[ 0.02081187 -0.04698074 -0.04354031 -0.0063564  -0.00389652 -0.04263849
  -0.03294184 -0.00977498 -0.01595835  0.04704111  0.03521866  0.02989128
   0.02634836  0.03439596 -0.01098412 -0.00317807]
 [ 0.03757856  0.04822851 -0.01892936 -0.00211356 -0.01231574 -0.01402745
  -0.02077849  0.04636785  0.04637593 -0.02264645  0.02440817  0.04189176
   0.00818812 -0.01063976  0.02517254 -0.0319974 ]
 [ 0.02135703  0.01097977  0.02606316  0.0438025   0.0305129   0.00850515
   0.01229875  0.0317629   0.00027193 -0.02940155  0.03526406 -0.0483651
   0.04935322 -0.00295262  0.00782329  0.00317358]
 [-0.01869469 -0.0293476   0.02265011  0.04520984  0.00571029  0.03589823
  -0.04632941  0.02564225  0.02615276 -0.04993917  0.04525671  0.04673379
  -0.03198602  0.0086054  -0.03958919 -0.03150535]
 [ 0.02608403 -0.00463984  0.0456433  -0.03808006 -0.00051624 -0.00292089
   0.04824655 -0.03491377  0.01601804 -0.04375848  0.0232757   0.036026
  -0.02450383 -0.02947688  0.00229589 -0.0491794

For every word we get the 16-dimension vector representation of that word

Now let's build a end-to-end workflow

In [34]:
import numpy as np

model = Sequential([
    # Input : Raw String
    layers.Input(shape = (1,), dtype = 'string'),

    # Vectorization
    vectorizer,

    # Embedding
    embedding,

    # RNN
    layers.SimpleRNN(
        units=RNN_UNITS,
        activation='relu',
        return_sequences=False,   # Does not return the output of the every hidden state
    ),

    # Output
    layers.Dense(1, activation = 'sigmoid')
])

model.summary()

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 8)              │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_13 (Embedding)        │ (None, 8, 16)          │        16,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,601 (68.75 KB)

 Trainable params: 17,601 (68.75 KB)

 Non-trainable params: 0 (0.00 B)

Let's try on real dataset

In [36]:
from keras.datasets import imdb
from keras.utils import pad_sequences

In [44]:
(X_train, y_train), (X_test, y_test) = imdb.load_data()

In [40]:
X_train.shape

(25000,)

In [41]:
X_train[0]

[1,
 14,
 22,
 16,
 43,
 530,
 973,
 1622,
 1385,
 65,
 458,
 4468,
 66,
 3941,
 4,
 173,
 36,
 256,
 5,
 25,
 100,
 43,
 838,
 112,
 50,
 670,
 22665,
 9,
 35,
 480,
 284,
 5,
 150,
 4,
 172,
 112,
 167,
 21631,
 336,
 385,
 39,
 4,
 172,
 4536,
 1111,
 17,
 546,
 38,
 13,
 447,
 4,
 192,
 50,
 16,
 6,
 147,
 2025,
 19,
 14,
 22,
 4,
 1920,
 4613,
 469,
 4,
 22,
 71,
 87,
 12,
 16,
 43,
 530,
 38,
 76,
 15,
 13,
 1247,
 4,
 22,
 17,
 515,
 17,
 12,
 16,
 626,
 18,
 19193,
 5,
 62,
 386,
 12,
 8,
 316,
 8,
 106,
 5,
 4,
 2223,
 5244,
 16,
 480,
 66,
 3785,
 33,
 4,
 130,
 12,
 16,
 38,
 619,
 5,
 25,
 124,
 51,
 36,
 135,
 48,
 25,
 1415,
 33,
 6,
 22,
 12,
 215,
 28,
 77,
 52,
 5,
 14,
 407,
 16,
 82,
 10311,
 8,
 4,
 107,
 117,
 5952,
 15,
 256,
 4,
 31050,
 7,
 3766,
 5,
 723,
 36,
 71,
 43,
 530,
 476,
 26,
 400,
 317,
 46,
 7,
 4,
 12118,
 1029,
 13,
 104,
 88,
 4,
 381,
 15,
 297,
 98,
 32,
 2071,
 56,
 26,
 141,
 6,
 194,
 7486,
 18,
 4,
 226,
 22,
 21,
 134,
 476,
 26,
 480,
 5

No need for vectorization because text is already preprocessed

In [45]:
X_train = pad_sequences(X_train, padding='post', maxlen=100)
X_test = pad_sequences(X_test, padding='post', maxlen=100)

In [46]:
X_train.shape

(25000, 100)

In [47]:
model =Sequential([
    layers.Input(shape=(100,)),
    layers.Embedding(10000, output_dim=EMBEDDING_DIM),
    layers.SimpleRNN(RNN_UNITS, activation='relu', return_sequences=False),
    layers.Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_14 (Embedding)        │ (None, 100, 16)        │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161,601 (631.25 KB)

 Trainable params: 161,601 (631.25 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.6996 - loss: 0.5618 - val_accuracy: 0.8083 - val_loss: 0.4268
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8497 - loss: 0.3553 - val_accuracy: 0.8377 - val_loss: 0.3786
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8295 - loss: 0.3713 - val_accuracy: 0.6962 - val_loss: 0.5505
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.8405 - loss: 0.3633 - val_accuracy: 0.8271 - val_loss: 0.4285
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9102 - loss: 0.2324 - val_accuracy: 0.8339 - val_loss: 0.4095
